# 指标计算模块演示 (Metrics Module)

本notebook演示hscredit库中指标计算模块的全部功能。

In [ ]:
# 添加项目路径
import sys
import os
sys.path.append('../')

# 初始化设置
from hscredit.utils import init_setting
init_setting(seed=42)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os

# 加载数据
data_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/hscredit.xlsx'
df = pd.read_excel(data_path)
print(f"数据形状: {df.shape}")

In [ ]:
# 定义目标列和排除列
target_col = 'FPD15'
exclude_cols = ['MOB1', 'MOB2', 'loan_date', 'FPD15', 'SFPD15']

# 获取特征列
feature_cols = [col for col in df.columns if col not in exclude_cols]
demo_feature = feature_cols[0]

# 准备数据
X = df[feature_cols]
y = df[target_col]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")

## 1. 导入指标计算模块

In [ ]:
from hscredit.core.metrics import (
    KS, AUC, Gini, PSI, IV,
    KS_bucket, ROC_curve,
    confusion_matrix, classification_report,
    PSI_table, CSI_table,
    IV_table,
    MSE, MAE, RMSE, R2
)

print("指标计算模块导入成功！")

## 2. 分类指标

KS、AUC、Gini等风控核心指标。

In [ ]:
from sklearn.linear_model import LogisticRegression
from hscredit.core.encoders import WOEEncoder

# WOE编码
woe = WOEEncoder()
X_train_woe = woe.fit_transform(X_train, y_train)
X_test_woe = woe.transform(X_test)

# 训练模型
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_woe, y_train)
y_pred_proba = lr.predict_proba(X_test_woe)[:, 1]

# 计算分类指标
ks_score = KS(y_test, y_pred_proba)
auc_score = AUC(y_test, y_pred_proba)
gini_score = Gini(y_test, y_pred_proba)

print("风控核心指标:")
print(f"  KS: {ks_score:.4f}")
print(f"  AUC: {auc_score:.4f}")
print(f"  Gini: {gini_score:.4f}")

## 3. 混淆矩阵和分类报告

In [ ]:
# 预测类别
y_pred = (y_pred_proba > 0.5).astype(int)

# 混淆矩阵
cm = confusion_matrix(y_test, y_pred)
print("混淆矩阵:")
print(cm)

# 分类报告
report = classification_report(y_test, y_pred)
print("\n分类报告:")
print(report)

## 4. 稳定性指标

PSI和CSI稳定性指标。

In [ ]:
# PSI计算
psi_score = PSI(y_train, y_test)
print(f"总体PSI: {psi_score:.4f}")

# 特征PSI
feature_psi = PSI_table(X_train[[demo_feature]], X_test[[demo_feature]])
print(f"\n{demo_feature} PSI分箱表:")
print(feature_psi)

In [ ]:
# 批量计算特征PSI
psi_results = {}
for col in feature_cols[:10]:
    try:
        psi = PSI(X_train[[col]], X_test[[col]])
        psi_results[col] = psi
    except:
        psi_results[col] = np.nan

psi_df = pd.Series(psi_results).sort_values(ascending=False)
print("特征PSI排名（前10）:")
print(psi_df)

## 5. 特征重要性指标

IV等信息价值指标。

In [ ]:
# 单特征IV
iv_score = IV(df[[demo_feature]], df[target_col])
print(f"{demo_feature} IV值: {iv_score:.4f}")

# IV分箱表
iv_table = IV_table(df[[demo_feature]], df[target_col])
print(f"\n{demo_feature} IV分箱表:")
print(iv_table)

In [ ]:
# 批量计算特征IV
iv_results = {}
for col in feature_cols[:10]:
    try:
        iv = IV(df[[col]], df[target_col])
        iv_results[col] = iv
    except:
        iv_results[col] = np.nan

iv_df = pd.Series(iv_results).sort_values(ascending=False)
print("特征IV排名（前10）:")
print(iv_df)

## 6. 分箱指标

WOE、IV、KS按分箱计算。

In [ ]:
from hscredit.core.binning import OptimalBinning
from hscredit.core.metrics.binning_metrics import compute_bin_stats, ks_by_bin, chi2_by_bin

# 分箱
binner = OptimalBinning(method='quantile', max_n_bins=5)
binner.fit(df[[demo_feature]], df[target_col])
X_binned = binner.transform(df[[demo_feature]], metric='indices')

# 计算分箱统计
bin_stats = compute_bin_stats(X_binned, df[target_col])
print(f"{demo_feature} 分箱统计:")
print(bin_stats)

# 分箱KS
ks_bins = ks_by_bin(X_binned, df[target_col])
print(f"\n分箱KS值:")
print(ks_bins)

## 7. 回归指标

MSE、MAE、RMSE、R2等回归评估指标。

In [ ]:
# 使用模型预测值作为示例
# 计算回归指标（这里用预测概率作为示例）
y_true_reg = y_test.astype(float)
y_pred_reg = y_pred_proba

mse_score = MSE(y_true_reg, y_pred_reg)
mae_score = MAE(y_true_reg, y_pred_reg)
rmse_score = RMSE(y_true_reg, y_pred_reg)
r2_score = R2(y_true_reg, y_pred_reg)

print("回归评估指标:")
print(f"  MSE: {mse_score:.4f}")
print(f"  MAE: {mae_score:.4f}")
print(f"  RMSE: {rmse_score:.4f}")
print(f"  R2: {r2_score:.4f}")

## 8. 指标汇总报告

将所有指标汇总保存。

In [ ]:
# 指标汇总
metrics_summary = pd.DataFrame({
    '指标': ['KS', 'AUC', 'Gini', 'PSI', 'MSE', 'MAE', 'RMSE'],
    '值': [ks_score, auc_score, gini_score, psi_score, mse_score, mae_score, rmse_score]
})

print("指标汇总:")
print(metrics_summary.to_string(index=False))

# 保存结果
output_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/metrics_results.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    metrics_summary.to_excel(writer, sheet_name='指标汇总', index=False)
    psi_df.to_frame('PSI值').to_excel(writer, sheet_name='特征PSI')
    iv_df.to_frame('IV值').to_excel(writer, sheet_name='特征IV')
    bin_stats.to_excel(writer, sheet_name='分箱统计', index=False)

print(f"\n指标结果已保存至: {output_path}")